In [ ]:
from pathlib import Path
import json
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_FILE = Path("nfhs_full5-30.csv")
GEO_FILE = Path("data/india_states.geojson")
META_FILE = Path("data/meta.json")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Publication defaults
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "serif",
    "font.size": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

df_nfhs = pd.read_csv(DATA_FILE)
with open(META_FILE) as f:
    meta = json.load(f)

ROUND_YEARS = meta["round_years"]
WAVE_MAP = {"NFHS-3": 3, "NFHS-4": 4, "NFHS-5": 5, "NFHS-6": 6}
YEAR_MID = {"NFHS-3": 2005.5, "NFHS-4": 2015.5, "NFHS-5": 2020.0, "NFHS-6": 2023.5}

df_nfhs.head()

In [ ]:
# Keywords
FERTILITY_KW   = ["fertility rate", "tfr", "total fertility",
                        "birth rate", "children born", "births per"]
PHONE_KW       = ["mobile phone", "mobile telephone", "cell phone",
                        "smartphone", "phone ownership", "use mobile"]
INTERNET_KW    = ["internet", "online"]
EDUCATION_KW   = ["literate", "literacy", "education", "schooling",
                        "no education", "secondary", "years of school"]
WEALTH_KW      = ["wealth", "poorest", "richest", "quintile",
                        "household assets", "below poverty"]
URBAN_KW       = ["urban", "rural"]
CONTRACEPTION_KW     = ["contracepti", "family planning", "modern method",
                        "unmet need", "steriliz"]
EMPOWERMENT_KW       = ["decision", "autonomy", "empowerment",
                        "own money", "bank account", "financial"]

# Harmonised indicator crosswalk (Pakhare & Joshi)
HARMONISED = {
    "tfr": "Total fertility rate (children per woman)",
    "mobile": "Women with own mobile phone (%)",
    "internet": "Women who ever used the internet (%)",
    "literacy": "Women (15-49) who are literate (%)",
    "schooling": "Women (15-49) with 10+ years schooling (%)",
    "fp_modern": "FP: Any modern method (%)",
    "unmet_fp": "Total unmet need for FP (%)",
    "empowerment": "Married women participate in HH decisions (%)",
    "electricity": "Households with electricity (%)",
    "bank": "Women with own bank/savings account (%)",
}

MAIN_ROUNDS = ["NFHS-4", "NFHS-5", "NFHS-6"]
INTERNET_ROUNDS = ["NFHS-5", "NFHS-6"]
REG_CONTROLS = ["schooling", "fp_modern", "electricity"]  # literacy absent in NFHS-6

EXCLUDE_STATES = {"India"}
# Legacy UTs split in NFHS-3/4; merged from NFHS-5 onward
LEGACY_UTS = {"Dadra and Nagar Haveli", "Daman and Diu"}

In [ ]:


def build_panel(df, rounds, indicators=None, area="Total", require_vars=None):
    """Wide state × round panel from harmonised indicators."""
    indicators = indicators or HARMONISED
    inv = {v: k for k, v in indicators.items()}
    require_vars = require_vars or ["tfr", "mobile"]

    sub = df[
        (df["area"] == area)
        & (~df["state"].isin(EXCLUDE_STATES))
        & (df["round"].isin(rounds))
        & (df["harmonized_indicator"].isin(indicators.values()))
    ].copy()
    sub = sub[~sub["data_flag"].astype(str).str.contains("Parenthesised", na=False)]

    wide = (
        sub.pivot_table(
            index=["state", "round"],
            columns="harmonized_indicator",
            values="value",
            aggfunc="first",
        )
        .rename(columns=inv)
        .reset_index()
    )
    wide["wave"] = wide["round"].map(WAVE_MAP)
    wide["year_mid"] = wide["round"].map(YEAR_MID)
    wide["year_label"] = wide["round"].map(ROUND_YEARS)

    req_cols = require_vars
    complete = wide.dropna(subset=req_cols)
    return wide, complete


panel_all, panel_main = build_panel(
    df_nfhs, MAIN_ROUNDS, require_vars=["tfr", "mobile"]
)
panel_reg = panel_main.dropna(subset=["tfr", "mobile"] + REG_CONTROLS)

panel_main.to_csv(OUTPUT_DIR / "panel_state_total.csv", index=False)
print(f"Main panel: {len(panel_main)} obs, {panel_main['state'].nunique()} states")
print(f"Regression sample: {len(panel_reg)} obs, {panel_reg['state'].nunique()} states")
panel_main.head()

In [ ]:
# --- Indicator audit: coverage matrix & keyword search ---

def coverage_matrix(df, var, rounds):
    tmp = df[df["round"].isin(rounds)].copy()
    tmp["has_val"] = tmp[var].notna().astype(int)
    return tmp.pivot_table(index="state", columns="round", values="has_val", aggfunc="max", fill_value=0)


print("=" * 60)
print("COVERAGE: TFR and Mobile (Total area, excl. India)")
print("=" * 60)
for label, var in [("TFR", "tfr"), ("Mobile", "mobile")]:
    mat = coverage_matrix(panel_all, var, MAIN_ROUNDS)
    print(f"\n{label} — states with data by round:")
    print(mat.sum(axis=0))
    missing = mat[(mat.sum(axis=1) < len(MAIN_ROUNDS))].index.tolist()
    if missing:
        print(f"  Incomplete states ({len(missing)}): {missing}")

lit_cov = coverage_matrix(panel_all, "literacy", MAIN_ROUNDS)
print(f"\nLiteracy NFHS-6 coverage: {lit_cov['NFHS-6'].sum()} states (0 expected — not harmonised in NFHS-6)")

print("\n" + "=" * 60)
print("ATTRITION")
print("=" * 60)
for desc, n in [
    ("All state-rounds (NFHS-4/5/6)", len(panel_all)),
    ("Non-missing TFR + mobile", len(panel_main)),
    ("+ non-missing controls", len(panel_reg)),
]:
    print(f"  {desc}: {n}")

def kw_match(text, keywords):
    t = str(text).lower()
    return any(k in t for k in keywords)

unmapped_phone = df_nfhs[
    (df_nfhs["area"] == "Total")
    & (~df_nfhs["state"].isin(EXCLUDE_STATES))
    & (df_nfhs["round"].isin(MAIN_ROUNDS))
    & (df_nfhs["harmonized_indicator"].isna() | (df_nfhs["harmonized_indicator"] == ""))
    & df_nfhs["indicator"].apply(lambda x: kw_match(x, PHONE_KW))
]["indicator"].unique()

print("\nUnmapped raw indicators matching PHONE_KW:")
print(unmapped_phone if len(unmapped_phone) else "  None — all phone measures are harmonised.")